# Durable Agent Runtime and MCP

Build a local review workflow with typed tool access, approvals, durable state, and replay-safe execution.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Practice case · The reviewer approves after the worker goes home

Jo coordinates public meeting agendas. A local tool can retrieve the outline, and a separate step records an approved note. Review can take hours. Jo needs the draft to survive a restart and wants a replay to be visible without creating another copy.

The workplace scenario is fictional.


## Connect and constrain


### Lecture 01 · Discover and call a local MCP tool

Trace the MCP initialization, tool discovery, and tool invocation over stdio.

#### Read the tool declaration before treating it as a promise

The tool advertises an agenda lookup and a declared identifier field. It does not promise that the requester can see every agenda.

Inspect the declared input and validate the result against a known case. Discovery tells the client what exists. Application policy still determines which calls are permitted.

#### How it works

A caller reaches a tool through a negotiated protocol and a declared input schema. With stdio transport, protocol messages use standard output and diagnostic messages use standard error. The notebook starts a minimal local fixture that offers a read-only agenda lookup.

##### Discover a contract, then inspect it

An MCP client establishes a session with a compatible server and requests its tool definitions. A definition can describe the shape of inputs and the tool's purpose. It is useful evidence about how to form a request. The application must still decide whether a tool is appropriate and verify that its behavior meets that contract.

In the local exercise, use the agenda lookup in read-only mode and compare the listed schema with the actual result. Record the server version and one known output. When debugging, distinguish a transport error, an unknown operation, an invalid argument, and a successful call that returned no useful evidence.

Treat external tool descriptions as data from another boundary. A readable description does not give the server authority to change the caller’s task. Keep the allowed operations narrow, test permission separately, and make the failure state legible to the reviewer.

#### Worked example

Initialize the session, discover lookup_agenda, and invoke it for a known meeting ID. Inspect the declared fields before sending arguments.

#### Try it

Run the discovery cell. Call lookup_agenda for the sample meeting and inspect the JSON-RPC result.


In [ ]:
import json
import subprocess
import sys

# Minimal local MCP server fixture. Each line of stdout is a protocol message.
server = """
import json
import sys

for line in sys.stdin:
    message = json.loads(line)
    method = message["method"]
    if method == "notifications/initialized":
        continue
    reply = {"jsonrpc": "2.0", "id": message["id"]}
    if method == "initialize":
        reply["result"] = {
            "protocolVersion": "2025-06-18",
            "capabilities": {"tools": {}},
            "serverInfo": {"name": "agenda-fixture", "version": "1.0"},
        }
    elif method == "tools/list":
        reply["result"] = {"tools": [{
            "name": "lookup_agenda",
            "description": "Read the local synthetic review agenda.",
            "inputSchema": {
                "type": "object", "properties": {"meeting": {"type": "string", "enum": ["M1"]}},
                "required": ["meeting"], "additionalProperties": False,
            },
        }]}
    elif method == "tools/call":
        params = message["params"]
        if params.get("name") == "lookup_agenda" and params.get("arguments") == {"meeting": "M1"}:
            reply["result"] = {"content": [{"type": "text", "text": "Review the accessibility checklist."}]}
        else:
            reply["error"] = {"code": -32602, "message": "Invalid tool or arguments"}
    else:
        reply["error"] = {"code": -32601, "message": "Method not found"}
    print(json.dumps(reply), flush=True)
"""


def exchange(method, params=None):
    messages = [
        {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {
            "protocolVersion": "2025-06-18", "capabilities": {},
            "clientInfo": {"name": "aiebok-workbook", "version": "1.0"},
        }},
        {"jsonrpc": "2.0", "method": "notifications/initialized"},
        {"jsonrpc": "2.0", "id": 2, "method": method, "params": params or {}},
    ]
    run = subprocess.run(
        [sys.executable, "-c", server], input="".join(json.dumps(msg) + "\n" for msg in messages),
        text=True, capture_output=True, timeout=5, check=True,
    )
    init, reply = [json.loads(line) for line in run.stdout.splitlines()]
    assert init["id"] == 1 and init["result"]["protocolVersion"] == "2025-06-18"
    assert reply["id"] == 2 and reply["jsonrpc"] == "2.0"
    return reply["result"]


tools = exchange("tools/list")["tools"]
assert [tool["name"] for tool in tools] == ["lookup_agenda"]
assert tools[0]["inputSchema"]["required"] == ["meeting"]
print("Discovered", tools)


**Compare your result**

The server returns one typed tool declaration and a structured read result over the local process boundary.


### Lecture 02 · Tool authorization and argument validation

Validate caller scope and arguments before dispatching a tool.

#### The same valid payload can be unauthorized

A reviewer and a guest supply the same meeting identifier. Both requests satisfy the input schema.

Check caller and resource policy in addition to argument shape. Record a denied request without sending it to the tool. This keeps a correctness check from being mistaken for authorization.

#### How it works

A schema constrains shape. Application policy decides who may call a tool and on which resource. Bound the tool surface to a known registry, and route invalid requests to a reviewable outcome.

##### Check both the argument and the actor

Validation asks whether a request has a well-formed resource identifier and expected fields. Authorization asks whether this caller can perform that operation for that resource. A guest and an approved reviewer can send identical valid payloads and should receive different outcomes.

Perform the authorization check in the trusted application or service boundary before dispatch. Use the caller identity provided by that boundary rather than a role asserted by generated text. Give read-only operations and persistent writes separate decisions. On a denial, return a controlled outcome with enough context for an operator to investigate.

Write paired tests for the same resource and different roles, plus a malformed request from an allowed role. The malformed case proves shape is checked. The paired case proves shape does not substitute for authority. Record the rejected reason without logging more request data than necessary.

#### Worked example

A reviewer may look up a team agenda. A guest with the same input shape has no permission, even when the tool is listed.

#### Try it

Run the allowed, denied, and invalid argument cases. Inspect where each decision is recorded.


In [ ]:
events = []


def lookup(role, meeting):
    if role != "reviewer":
        events.append("denied")
        return None
    if meeting != "M1":
        events.append("invalid_arguments")
        return None
    result = exchange("tools/call", {"name": "lookup_agenda", "arguments": {"meeting": meeting}})
    events.append("read")
    return result["content"][0]["text"]


assert lookup("guest", "M1") is None
assert lookup("reviewer", "M9") is None
assert lookup("reviewer", "M1") == "Review the accessibility checklist."
assert events == ["denied", "invalid_arguments", "read"]


**Compare your result**

Only a caller with the approved role and valid meeting ID reaches the tool.


## Pause, recover and replay


### Lecture 03 · Persistent approval and checkpoint recovery

Persist a proposal and its approval against a payload fingerprint, then reopen the runtime.

#### Persist what the reviewer actually saw

Jo approves a draft after the workflow restarts. A corrected draft exists with a different payload.

Persist the proposed payload and its fingerprint before pause. Bind the human decision to exactly that version. A changed payload should be reviewed anew.

#### How it works

Persist pending work before waiting for a reviewer. Bind approval to the specific proposal so a modified payload needs review. On restart, reconstruct the next permitted transition from stored state.

##### Persist the state a person saw when they approved

A waiting workflow needs data that can survive restart: the proposed effect, its exact revision, who can approve it, the recorded approval state, and the next legal transition. If the worker only holds that state in memory, a restart can transform an approved or pending effect into an ambiguous one.

Bind approval to the reviewed version. In the agenda exercise, change the note after storing a pending request and verify that approval for the old note cannot apply to the revision. Restore a checkpoint in a fresh process and confirm it reports whether the workflow is still waiting.

Approval and completion are distinct facts. An approval authorizes an effect under specified conditions. A completion record reports that the effect happened. If an operation can succeed while a response is lost, retain a stable identifier and a reconciliation route. Resume only after you can tell which facts are already durable.

#### Worked example

A reviewer approves a note for meeting M1. Editing the text after review changes its fingerprint, so the previous approval no longer applies.

#### Try it

Reopen the SQLite store after creating a proposal. Verify that pending work stays pending and compare a mismatched approval.


In [ ]:
from hashlib import sha256
from pathlib import Path
import sqlite3
from tempfile import TemporaryDirectory


def open_store(path):
    db = sqlite3.connect(path)
    db.executescript("""
        CREATE TABLE IF NOT EXISTS proposals (
          id TEXT PRIMARY KEY, payload TEXT NOT NULL, digest TEXT NOT NULL, approved INTEGER NOT NULL
        );
        CREATE TABLE IF NOT EXISTS notes (id TEXT PRIMARY KEY, payload TEXT NOT NULL);
    """)
    return db


def propose(db, identifier, payload):
    digest = sha256(payload.encode()).hexdigest()
    with db:
        db.execute("INSERT INTO proposals VALUES (?, ?, ?, 0)", (identifier, payload, digest))
    return digest


def approve(db, identifier, digest):
    with db:
        return db.execute("UPDATE proposals SET approved = 1 WHERE id = ? AND digest = ?",
                          (identifier, digest)).rowcount == 1


directory = TemporaryDirectory(prefix="aiebok-review-")
database = Path(directory.name) / "review.sqlite"
db = open_store(database)
digest = propose(db, "proposal-1", "M1 has completed accessibility review")
db.close()
db = open_store(database)
assert db.execute("SELECT approved FROM proposals").fetchone() == (0,)
assert not approve(db, "proposal-1", "different-payload")
assert db.execute("SELECT approved FROM proposals").fetchone() == (0,)


**Compare your result**

The record survives a restart and the wrong fingerprint leaves it awaiting approval.


### Lecture 04 · Idempotent execution and decision traces

Commit a local effect exactly once across retries and distinguish external effects.

#### A replay is neither a new note nor a new approval

A write completes and the worker is interrupted before the caller sees success. On restart, the same command arrives.

Commit the local note and its completion record atomically. Use the command identity to return the existing result on replay. For external effects, rely on the external idempotency contract or reconcile before trying again.

#### How it works

Record the action and its state transition in the same transaction for a local database effect. A retry can then find the completed record. A remote effect needs its own idempotency contract or a reconciliation process.

##### Define what a replay is allowed to do

An idempotent operation can accept repeated requests for the same intended effect without producing a second effect. Use a stable identity tied to the logical operation, not a new identifier for each delivery attempt. With a local database, committing a note and its completion record together gives the next worker one place to inspect.

A remote service may complete a write before the caller times out. Local memory alone cannot tell whether it happened. Use the service's idempotency contract where possible, otherwise reconcile before retrying. When evidence still cannot establish what happened, stop and ask an operator instead of sending a second notification blindly.

Log the operation identity, reviewed revision, effect state, and retry disposition. Verify a successful repeat returns the prior result and a genuinely changed request needs a new decision. The trace should explain why the system resumed, waited, or declined to repeat the action.

#### Worked example

The worker restarts after saving an approved agenda note and before returning success. The next attempt reports the existing completion.

#### Try it

Reopen the store again and replay the approved action. Check the persistent effect count and trace.


In [ ]:
def execute(db, identifier):
    with db:
        db.execute("BEGIN IMMEDIATE")
        row = db.execute("SELECT payload, approved FROM proposals WHERE id = ?", (identifier,)).fetchone()
        if row is None or not row[1]:
            return "awaiting_approval"
        effect = db.execute("INSERT OR IGNORE INTO notes VALUES (?, ?)", (identifier, row[0]))
        return "completed" if effect.rowcount else "replayed"


assert execute(db, "proposal-1") == "awaiting_approval"
assert approve(db, "proposal-1", digest)
assert execute(db, "proposal-1") == "completed"
db.close()
db = open_store(database)
try:
    assert execute(db, "proposal-1") == "replayed"
    assert db.execute("SELECT count(*) FROM notes").fetchone() == (1,)
    print("Decision trace", events + ["awaiting_approval", "approved", "completed", "replayed"])
finally:
    db.close()
    directory.cleanup()


**Compare your result**

The replay reports the existing result and there is one note. Explain why this guarantee would change for a remote calendar API.


## Course project

Build a supervised agenda-review workflow using an MCP-compatible local tool. Show validated calls, approval, a process restart, and no duplicate effect.

Capture a tool discovery transcript, a denied tool call, a human approval, and a replay from reopened state. Document which guarantees depend on the local transaction.
